#Assignment 6
Use the notebook from lecture 10 on RAGs. Manipulate a few parts of the RAG pipeline (hint: try changing the chunk size, chunk overlap, use a different splitter, manipulate the number of splits retrieved and how they are retrieved or similar things.).    

Does your change in configuration improve or degrade your RAG in some way (e.g. better/worse output, fewer/more tokens used, faster/slower run time)?

* Briefly reflect on your results

In [ ]:
!pip install -q langchain langchain_community langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.0/607.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.7/407.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.0 MB/s eta 0:

In [ ]:
from google.colab import userdata
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')
project_name = "Assignment6"  # Update with your project name
os.environ["LANGCHAIN_PROJECT"] = project_name  # Optional: "default" is used if not set

In [ ]:
!pip install -qU langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 19.4 MB/s eta 0:00:00


In [ ]:
!pip install langchainhub

In [ ]:
!pip install langsmith

In [ ]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = userdata.get('testkey')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

# Load, chunk and index the contents of the blog.
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=150,
                                              chunk_overlap=5)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# Retrieve and generate using the relevant snippets of the blog.
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
prompt = hub.pull("rlm/rag-prompt")


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
).with_config({"run_name": "6th run"})

rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a complicated task into smaller, manageable subgoals to enable more efficient handling. This approach allows agents to plan ahead and tackle complex tasks step-by-step. By doing so, it enhances understanding and execution of the overall task.'

In [ ]:
for chunk in rag_chain.stream("What is Task Decomposition?"):
    print(chunk, end="", flush=True)

Task Decomposition is the process of breaking down a complex task into smaller, manageable steps. This can be accomplished using techniques like Chain of Thought (CoT), where models are prompted to "think step by step," or through task-specific instructions. It allows for better planning and understanding of the task at hand.

In [ ]:
# cleanup
vectorstore.delete_collection()

##First run - RecursiveCharacterTextSplitter
* chunk_size = 1000
* chunk_overlap = 200

**Text output:**
Task decomposition is the process of breaking down complex tasks into smaller, manageable steps to facilitate planning and execution. This is often achieved using techniques like Chain of Thought (CoT), where the model is prompted to "think step by step," or through methods such as Tree of Thoughts, which explores multiple reasoning paths. It can be performed with simple prompts, task-specific instructions, or human guidance.

##Second run - RecursiveCharacterTextSplitter
* chunk_size = 500
* chunk_overlap = 100

**Text output:**

Task Decomposition is a technique that involves breaking down complex tasks into smaller, more manageable steps. This is often achieved through prompting methods such as instructing a model to "think step by step" or providing task-specific instructions. It enhances model performance and helps interpret the model's reasoning process.

##Third run - RecursiveCharacterTextSplitter
* chunk_size = 200
* chunk_overlap = 25

**Text output:**

Task Decomposition is the process of breaking down large tasks into smaller, manageable subgoals to facilitate efficient handling of complex tasks. It can be achieved through techniques such as prompting, task-specific instructions, or human inputs. This approach enhances model performance by allowing for a step-by-step thought process.

##4th run - Charactertextsplitter
* chunk_size = 400
* chunk_overlap = 10
* Retriever - k = 7

**Text output**
Task Decomposition is the process of breaking down large tasks into smaller, manageable subgoals, which facilitates efficient handling of complex tasks. It can be achieved through methods like prompting models to think step by step or using task-specific instructions. This approach enhances performance and provides insight into the model's reasoning.

##5th run - Charactertextsplitter
* chunk_size = 5000
* chunk_overlap = 150
* retriever - k = 4

**Text output**
Task Decomposition involves breaking down large tasks into smaller, manageable subgoals to simplify complex tasks. This can be achieved through various methods, including prompting language models or using task-specific instructions. It enhances efficiency and clarity in task execution.

#6th run - RecursiveCharacterTextSplitter
* chunk_size = 150
* chunk_overlap = 5
* retriever - k = 10

Task decomposition is the process of breaking down a complicated task into smaller, manageable subgoals to enable more efficient handling. This approach allows agents to plan ahead and tackle complex tasks step-by-step. By doing so, it enhances understanding and execution of the overall task.